# Notebook 9b: Do near-copy resumes inflate the "resume text alone" result?

**Runtime:** CPU, about 5 minutes. Needs Notebook 9's output (`results/09_review_checks/A1_near_duplicate_pairs.csv`).

**Why:** Notebook 9 found that 148 of the 643 resumes have a near-copy (78 pairs, many word-for-word identical with tiny character differences). The IDs come from the exact text, so near-copies count as *different* resumes. Notebooks 6b and 6c tested the resume-text-only classifier on "unseen resumes" by grouping folds by **resume ID**, so a near-copy of a test resume could still be in training. Here we group by **near-copy group** instead (a resume and all its near-copies always sit in the same fold).

**Rule fixed before running (ANALYSIS-PLAN addendum 2026-10-19b):** the paper keeps the claim "resume text alone predicts Good vs Potential on unseen resumes" if the per-job AUC over the 91 mixed jobs, grouped by near-copy group, has a 95% CI above 0.5. Otherwise the claim is withdrawn and 6.3 says the 6b number was inflated by near-copies. The tech / non-tech numbers are updated either way.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import re, difflib
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer

DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
OUT = f'{RESULTS}/09_review_checks'
SEED = 42

all_df = pd.concat([pd.read_parquet(f'{DATA}/{s}.parquet') for s in ('train', 'val', 'test')], ignore_index=True)
res_text = all_df.drop_duplicates('resume_id').set_index('resume_id').resume_text
job_text = all_df.drop_duplicates('job_id').set_index('job_id').job_text
near = pd.read_csv(f'{OUT}/A1_near_duplicate_pairs.csv')

def groups(ids, p):
    pos = {x: i for i, x in enumerate(ids)}
    g = sparse.coo_matrix((np.ones(len(p)), (p.a.map(pos), p.b.map(pos))), shape=(len(ids), len(ids)))
    n, comp = connected_components(g, directed=False)
    return n, pd.Series(comp, index=ids)

n_res, res_group = groups(list(res_text.index), near[near.texts == 'resumes'])
n_job, job_group = groups(list(job_text.index), near[near.texts == 'jobs'])
print(f'resumes: {len(res_text)} IDs -> {n_res} near-copy groups; largest group {res_group.value_counts().max()}')
print(f'jobs:    {len(job_text)} IDs -> {n_job} near-copy groups; largest group {job_group.value_counts().max()}')
print('resume group sizes:', res_group.value_counts().value_counts().sort_index().to_dict())

## 1. What differs between "identical" resumes?
For the five most similar resume pairs, the first few places where the texts differ. One line from this goes into the paper (for example "differ only in punctuation or a few words").

In [ ]:
rp = near[near.texts == 'resumes'].sort_values(['cosine', 'jaccard'], ascending=False).head(5)
for _, r in rp.iterrows():
    a, b = res_text[r.a], res_text[r.b]
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    diffs = [(op, a[i1:i2][:40], b[j1:j2][:40]) for op, i1, i2, j1, j2 in sm.get_opcodes() if op != 'equal']
    print(f'cosine {r.cosine:.3f}  lengths {len(a)} / {len(b)} chars  similarity ratio {sm.ratio():.3f}  differences: {len(diffs)}')
    for d in diffs[:4]:
        print('    ', d)
    print()

## 2. Do near-copies get the same labels?
If Good vs Potential reflects the resume, near-copies of one resume should get similar shares of Good. We compare each near-copy pair's share of Good (among its fit pairs) with random resume pairs.

In [ ]:
fit_all = all_df[all_df.relevance > 0]
share = fit_all.assign(good=(fit_all.relevance == 2)).groupby('resume_id').good.agg(['mean', 'size'])
share = share[share['size'] >= 2]['mean']
rp_all = near[near.texts == 'resumes']
d_near = [abs(share[a] - share[b]) for a, b in zip(rp_all.a, rp_all.b) if a in share and b in share]
r = np.random.default_rng(SEED); ids = share.index.to_numpy()
d_rand = [abs(share[x] - share[y]) for x, y in zip(r.choice(ids, 5000), r.choice(ids, 5000)) if x != y]
print(f'near-copy pairs with 2+ fit pairs each: {len(d_near)}; mean |difference in share Good| = {np.mean(d_near):.3f} '
      f'vs {np.mean(d_rand):.3f} for random resume pairs')

## 3. The resume-text-only classifier, grouped by resume ID (6b/6c) and by near-copy group
First row of each pair must reproduce 6c: all mixed jobs 0.635, tech mixed 0.666, non-tech mixed 0.573, tech words removed 0.602.

In [ ]:
fit = all_df[all_df.relevance > 0].copy().reset_index(drop=True)
fit['good'] = (fit.relevance == 2).astype(int)
fit['resume_group'] = fit.resume_id.map(res_group)

# tech vocabulary and job groups copied from Notebook 6c
TECH = {
    'software': r'\bsoftware\b', 'developer': r'\bdevelopers?\b', 'programming': r'\bprogramm(?:er|ers|ing)\b',
    'javascript': r'\bjavascript\b', 'typescript': r'\btypescript\b', 'python': r'\bpython\b', 'java': r'\bjava\b',
    'sql': r'\bsql\b', 'mysql': r'\bmysql\b', 'postgres': r'\bpostgre(?:s|sql)\b', 'mongodb': r'\bmongo(?:db)?\b',
    'html': r'\bhtml5?\b', 'css': r'\bcss3?\b', 'php': r'\bphp\b', '.net': r'(?:asp)?\.net\b', 'c#': r'\bc#',
    'c++': r'\bc\+\+', 'react': r'\breact(?:\.?js)?\b', 'angular': r'\bangular(?:js)?\b', 'node': r'\bnode\.?js\b',
    'django': r'\bdjango\b', 'aws': r'\baws\b', 'azure': r'\bazure\b', 'cloud': r'\bcloud\b', 'devops': r'\bdevops\b',
    'docker': r'\bdocker\b', 'kubernetes': r'\bkubernetes\b', 'linux': r'\blinux\b', 'database': r'\bdatabases?\b',
    'data science/analytics': r'\bdata (?:scien\w*|analy\w*|engineer\w*|warehous\w*)',
    'machine learning': r'\bmachine learning\b|\bdeep learning\b', 'network admin': r'\bnetwork (?:admin\w*|engineer\w*|security)\b',
    'cybersecurity': r'\bcyber ?security\b', 'it support': r'\bit (?:support|specialist|infrastructure)\b|\bhelp ?desk\b',
    'information technology': r'\binformation technology\b', 'full stack': r'\bfull[- ]?stack\b',
    'front end': r'\bfront[- ]?end\b', 'back end': r'\bback[- ]?end\b', 'web': r'\bweb (?:develop\w*|design\w*|applications?)\b',
    'api': r'\b(?:rest(?:ful)? )?apis?\b', 'git': r'\bgit(?:hub)?\b', 'qa/testing': r'\bselenium\b|\bqa (?:engineer|tester|analyst)\b',
    'bi tools': r'\btableau\b|\bpower ?bi\b', 'sql server': r'\bsql server\b',
}
TECH_RE = {k: re.compile(v, re.I) for k, v in TECH.items()}
def tech_count(t): return sum(1 for rx in TECH_RE.values() if rx.search(t))
def strip_tech(t):
    for rx in TECH_RE.values(): t = rx.sub(' ', t)
    return t

job_rate = fit.groupby('job_id').good.mean()
mixed_ids = job_rate.index[(job_rate > 0) & (job_rate < 1)]
tech_ids = set(job_text.index[job_text.map(tech_count) >= 3])
print(f'{len(mixed_ids)} mixed jobs, {len([j for j in mixed_ids if j in tech_ids])} of them tech')

def grouped_oof(df, texts, group_col):
    y = df.good.to_numpy(); oof = np.zeros(len(df)); texts = np.asarray(texts, dtype=object)
    for tr, te in GroupKFold(5).split(df, y, df[group_col]):
        vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=50_000)
        Xtr = vec.fit_transform(texts[tr]); Xte = vec.transform(texts[te])
        clf = LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced').fit(Xtr, y[tr])
        oof[te] = clf.predict_proba(Xte)[:, 1]
    return oof

def per_job_auc(df, col):
    out = {}
    for jid, g in df.groupby('job_id'):
        if 0 < g.good.sum() < len(g):
            out[jid] = roc_auc_score(g.good, g[col]) if g[col].nunique() > 1 else 0.5
    return pd.Series(out)

def boot_ci(v, n=2000):
    r = np.random.default_rng(SEED); v = np.asarray(v)
    b = np.array([r.choice(v, len(v)).mean() for _ in range(n)])
    return np.percentile(b, [2.5, 97.5]), min(1.0, 2 * min((b <= 0.5).mean(), (b >= 0.5).mean()))

stripped = fit.resume_text.map(strip_tech)
rows = []
for scorer, texts in [('resume text only', fit.resume_text), ('resume text only, tech words removed', stripped)]:
    for grouping in ['resume_id', 'resume_group']:
        col = f'{scorer} | {grouping}'
        fit[col] = grouped_oof(fit, texts, grouping)
        a = per_job_auc(fit[fit.job_id.isin(mixed_ids)], col)
        for grp, ids in [('all mixed jobs', a.index), ('tech mixed jobs', [j for j in a.index if j in tech_ids]),
                         ('NON-tech mixed jobs', [j for j in a.index if j not in tech_ids])]:
            v = a.loc[list(ids)]; (lo, hi), p = boot_ci(v)
            rows.append({'scorer': scorer, 'folds grouped by': 'resume ID (6b/6c)' if grouping == 'resume_id' else 'near-copy group',
                         'jobs': grp, 'n jobs': len(v), 'per-job AUC': round(v.mean(), 3),
                         'CI': f'[{lo:.3f}, {hi:.3f}]', 'p vs 0.5': float(f'{p:.3g}')})
tab = pd.DataFrame(rows)
display(tab)

ref = tab[(tab['folds grouped by'] == 'resume ID (6b/6c)')].set_index(['scorer', 'jobs'])['per-job AUC']
exp = {('resume text only', 'all mixed jobs'): 0.635, ('resume text only', 'tech mixed jobs'): 0.666,
       ('resume text only', 'NON-tech mixed jobs'): 0.573, ('resume text only, tech words removed', 'all mixed jobs'): 0.602}
bad = {k: ref[k] for k, v in exp.items() if abs(ref[k] - v) > 0.0015}
print('Reproduction of 6c:', 'OK' if not bad else f'MISMATCH {bad}  <- stop and')

main = tab[(tab.scorer == 'resume text only') & (tab['folds grouped by'] == 'near-copy group') & (tab.jobs == 'all mixed jobs')].iloc[0]
lo = float(main.CI.strip('[]').split(',')[0])
print(f"RULE: near-copy grouped per-job AUC {main['per-job AUC']} {main.CI} ->",
      'claim KEPT (CI above 0.5)' if lo > 0.5 else 'claim WITHDRAWN (CI reaches 0.5) ->')

In [ ]:
tab.to_csv(f'{OUT}/9b_resume_text_only_nearcopy.csv', index=False)
pd.DataFrame({'texts': ['resumes', 'jobs'], 'ids': [len(res_text), len(job_text)], 'near-copy groups': [n_res, n_job]}).to_csv(
    f'{OUT}/9b_effective_counts.csv', index=False)
print('saved')